In [1]:
BATCH_SIZE = 16
LABEL= "Friendly"

In [2]:
%load_ext autoreload
%autoreload 2
import os
from hireverse.utils.dataset_handler import DatasetHandler
from hireverse.utils.utils import BASE_DIR

participant_ids = DatasetHandler.get_participant_ids()

In [3]:
import gc
import cv2
import numpy as np
from tensorflow.keras.utils import to_categorical


def participant_frames_generator(participant_id, number_of_frames_in_batch=4):
    label = DatasetHandler.get_labels_dict(participant_id)[LABEL]
    total_frames = DatasetHandler.get_number_of_frames(participant_id)
    frame_yielder = DatasetHandler.yield_sorted_participant_frames_images(participant_id, is_image_greyscale=True)
    
    for j in range(0, total_frames, number_of_frames_in_batch):
        frames_batch = []
        for i in range(number_of_frames_in_batch):
            try:
                frame = next(frame_yielder)
                frame = frame.astype('float32') / 255.0  # Normalize
                frames_batch.append(frame)
            except StopIteration:
                break
        
        # Yield only if we have at least 1 frame
        if frames_batch:
            yield frames_batch, label


def to_100class(score_1_to_10):
    return np.floor((score_1_to_10 - 1) * 10 + np.random.uniform(0, 10))

def train_generator(train_ids, number_of_videos_in_the_batch=2):
    # Initialize the frame generators for each participant dynamically
    participants_frame_gens = {participant_id: participant_frames_generator(participant_id) for participant_id in train_ids}
    
    while True:  # Loop indefinitely for continuous training
        selected_ids = np.random.choice(train_ids, size=number_of_videos_in_the_batch, replace=False)
        X_train = []
        y_train = []
        
        for participant_id in selected_ids:
            participant_frame_gen = participants_frame_gens[participant_id]
            try:
                frames_batch, label = next(participant_frame_gen)
                X_train.append(frames_batch)
                y_train.extend([to_100class(label)] * len(frames_batch))
            except StopIteration:
                # Skip exhausted generators (move to next participant)
                continue
        
        if X_train:
            X_train = np.concatenate(X_train, axis=0)  # shape: (96, 640, 640)
            
            y_train = np.array(y_train)  # shape: (96, ...)
            yield X_train, y_train

In [4]:
from sklearn.model_selection import train_test_split

# TODO: use group split
train_ids, temp_ids = train_test_split(participant_ids, test_size=0.5, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.5, random_state=42)

train_gen = train_generator(train_ids)

In [5]:
X, y = next(train_gen)
print(X.shape)
print(y.shape)

(8, 640, 640)
(8,)


In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_competency_cnn(input_shape=(640, 640, 1)):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(100, activation='softmax')  # Single competency head
    ])
    model.compile(optimizer='adam',
                 loss='sparse_categorical_crossentropy',
                 metrics=['accuracy'])
    return model

# Initialize model
model = build_competency_cnn()
model.summary()

/Users/bassel27/personal_projects/hireverse/venv/lib/python3.9/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 638, 638, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 319, 319, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 317, 317, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 158, 158, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 156, 156, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        12,900 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,572 (412.39 KB)

 Trainable params: 105,572 (412.39 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
model.fit(
    train_generator(train_ids),
    steps_per_epoch=len(train_ids) // 6,
    epochs=10
)

Epoch 1/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 11s 698ms/step - accuracy: 0.0000e+00 - loss: 4.5997
Epoch 2/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 675ms/step - accuracy: 0.0671 - loss: 4.5330
Epoch 3/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 8s 723ms/step - accuracy: 0.0000e+00 - loss: 4.5500
Epoch 4/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 672ms/step - accuracy: 0.0000e+00 - loss: 3.9504
Epoch 5/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 617ms/step - accuracy: 0.0655 - loss: 3.9525
Epoch 6/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 652ms/step - accuracy: 0.0000e+00 - loss: 3.9713
Epoch 7/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 655ms/step - accuracy: 0.0216 - loss: 3.8836
Epoch 8/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 599ms/step - accuracy: 0.0000e+00 - loss: 3.6709
Epoch 9/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 687ms/step - accuracy: 0.0532 - loss: 3.5993
Epoch 10/10
11/11 ━━━━━━━━━━━━━━━━━━━━ 8s 691ms/step - accuracy: 0.0216 - loss: 3.7472
